# SilvanStrategy — Frozen Backtest Analysis

**© 2026 Alessandro Arrabito** · [Backtests, not Signals](https://backtestsnotsignals.substack.com)

---

> ⚠️ **Educational example. Not financial advice.** SilvanStrategy is **intentionally overfit** — a
> deliberate case study on why so many public freqtrade strategies look great in a backtest and fail
> live. It should not be traded. Full writeup: *"Freqtrade Strategy: +983% in 6 Years (Full Backtest
> Inside)"* on [Backtests, not Signals](https://backtestsnotsignals.substack.com), code and config on
> [GitHub](https://github.com/arrabyte/freqtrade-strategies).

## What this notebook does

Every chart and every number in the article comes from running real freqtrade backtests. This notebook
loads the **frozen results of those exact runs** (checked into this repo under `data/`, alongside the
`adx_threshold x bb_std` parameter-sweep CSV) and reproduces them deterministically — no re-fetching data
from an exchange, no re-running hyperopt, no risk of a different result next time you run this notebook.

If you'd rather reproduce everything from scratch — including re-downloading the OHLCV/funding/mark data
and re-running the backtest yourself — see the `freqtrade backtesting` command in the repo README. This
notebook is the frozen, always-reproducible version of the same result.

Four backtest results are used here, all produced with the exact strategy code in
[`user_data/strategies/SilvanStrategy.py`](../strategies/SilvanStrategy.py):

| File | What it is |
|---|---|
| `silvan_headline_full_period.zip` | The headline result: full 2018–2026 window, parameters hyperopted on that same window (the "+983%" backtest). |
| `silvan_splitA_full_period.zip` | Parameters hyperopted on 2018–2023 only, then run **unmodified** across the full 2018–2026 window. |
| `silvan_splitB_full_period.zip` | Parameters hyperopted on 2018–2024 only, then run **unmodified** across the full 2018–2026 window. |
| `silvan_splitC_full_period.zip` | Parameters hyperopted on 2018–2022 only, then run **unmodified** across the full 2018–2026 window. |

For splits A/B/C, in-sample and out-of-sample metrics below are computed by slicing each *single*
continuous run at its own train/test cutoff date — a cleaner, single-source-of-truth way to compute a
walk-forward split than running the in-sample and out-of-sample windows as two separate isolated
backtests (which is what produced the numbers quoted in the article). The two methods don't match
exactly: stake sizing is `unlimited` (a fraction of the *current* wallet balance), so an isolated
OOS-only run starting fresh at 1,000 USDT admits a different sequence of trades — capped by the same 5
open-trade limit — than a continuous run entering the OOS window already compounded to several times
that. Most metrics below land within a point or two of the article's numbers; out-of-sample CAGR for
Split A is the outlier, coming out slightly *negative* here (-1.32%) against the article's +3.65%.
Either way the conclusion is the same, if anything more so: same severe in-sample vs. out-of-sample
collapse on every split, computed a second, independent way.

In [1]:
import io
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.float_format", "{:.4f}".format)

DATA_DIR = Path("data")
STRATEGY = "SilvanStrategy"

HEADLINE_ZIP = DATA_DIR / "silvan_headline_full_period.zip"
SPLIT_A_ZIP  = DATA_DIR / "silvan_splitA_full_period.zip"
SPLIT_B_ZIP  = DATA_DIR / "silvan_splitB_full_period.zip"
SPLIT_C_ZIP  = DATA_DIR / "silvan_splitC_full_period.zip"
HEATMAP_CSV  = DATA_DIR / "silvan_param_sweep_adx_bbstd.csv"

STARTING_BALANCE = 1000.0

print("Data files present:", all(p.exists() for p in [HEADLINE_ZIP, SPLIT_A_ZIP, SPLIT_B_ZIP, SPLIT_C_ZIP, HEATMAP_CSV]))

Data files present: True


In [2]:
# Load trades + wallet equity curve from a frozen freqtrade backtest-result zip.
def load_backtest(zip_path: Path, strategy_key: str = STRATEGY):
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        json_name = next(n for n in names if n.endswith(".json") and "meta" not in n)
        with zf.open(json_name) as f:
            raw = json.load(f)

        wallet_name = next((n for n in names if "_wallet" in n and n.endswith(".feather")), None)
        wallet_df = None
        if wallet_name:
            with zf.open(wallet_name) as f:
                wallet_df = pd.read_feather(io.BytesIO(f.read()))

        mktchg_name = next((n for n in names if "_market_change" in n and n.endswith(".feather")), None)
        mktchg_df = None
        if mktchg_name:
            with zf.open(mktchg_name) as f:
                mktchg_df = pd.read_feather(io.BytesIO(f.read()))

    strat_data = raw["strategy"][strategy_key]
    return strat_data, wallet_df, mktchg_df


def build_trades(strat_data):
    trades = pd.DataFrame(strat_data["trades"])
    trades["open_date"] = pd.to_datetime(trades["open_date"])
    trades["close_date"] = pd.to_datetime(trades["close_date"])
    trades["win"] = trades["profit_ratio"] > 0
    return trades.sort_values("close_date").reset_index(drop=True)


def equity_from_trades(trades, starting_balance=STARTING_BALANCE):
    eq = starting_balance + trades["profit_abs"].cumsum()
    dd = (eq - eq.cummax()) / eq.cummax() * 100
    return eq, dd


def compute_metrics(wallet_df, label="Strategy"):
    # Mirrors freqtrade/data/metrics.py exactly (calculate_cagr, calculate_sharpe_from_balance,
    # calculate_sortino_from_balance, calculate_calmar_from_balance) so these numbers match what
    # `freqtrade backtesting` itself would print for the same balance history. Calmar in
    # particular is NOT the textbook CAGR/MaxDD: freqtrade uses
    # (mean daily simple return % / relative drawdown) * sqrt(365).
    w = wallet_df.copy()
    w["date"] = pd.to_datetime(w["date"]).sort_values()
    w = w.sort_values("date").reset_index(drop=True)

    starting_balance = float(w["total_quote"].iloc[0])
    final_balance = float(w["total_quote"].iloc[-1])
    days_period = max(1, (w["date"].iloc[-1] - w["date"].iloc[0]).days)

    total_pct = (final_balance / starting_balance - 1) * 100
    cagr = ((final_balance / starting_balance) ** (365 / days_period) - 1) * 100

    # Daily returns: resample to end-of-day first (wallet snapshots are sub-daily).
    daily_balance = w.set_index("date")["total_quote"].resample("1D").last().dropna()
    daily_ret = daily_balance.pct_change().dropna()

    def annualized_ratio(mean, denom):
        return float(mean / denom * np.sqrt(365)) if denom not in (0, None) and not np.isnan(denom) else -100.0

    sharpe = annualized_ratio(daily_ret.mean(), daily_ret.std(ddof=0)) if len(daily_ret) else 0.0
    downside = daily_ret[daily_ret < 0]
    sortino = annualized_ratio(daily_ret.mean(), downside.std(ddof=0)) if len(daily_ret) else 0.0

    # Relative account drawdown: (running high - balance) / running high, as a positive fraction.
    high_water = w["total_quote"].cummax()
    rel_dd_series = (high_water - w["total_quote"]) / high_water
    max_rel_dd = rel_dd_series.max()

    total_profit_frac = (final_balance - starting_balance) / starting_balance
    expected_returns_mean = total_profit_frac / days_period * 100
    calmar = annualized_ratio(expected_returns_mean, max_rel_dd)

    return {
        "Label": label,
        "Total %": f"{total_pct:+.2f}%",
        "CAGR %": f"{cagr:+.2f}%",
        "Sharpe": f"{sharpe:.2f}",
        "Sortino": f"{sortino:.2f}",
        "Max DD %": f"{-max_rel_dd * 100:.2f}%",
        "Calmar": f"{calmar:.2f}",
    }

## 1. The headline backtest

Hyperopted on the full 2018–2026 window, reported on that same window — sin #2 in the article. This is the equity curve at the top of the post.

In [3]:
headline_data, headline_wallet, headline_mktchg = load_backtest(HEADLINE_ZIP)
headline_trades = build_trades(headline_data)
headline_eq, headline_dd = equity_from_trades(headline_trades)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3],
                     subplot_titles=["SilvanStrategy — Equity Curve (headline, full-period fit)", "Drawdown %"])
fig.add_trace(go.Scatter(x=headline_trades["close_date"], y=headline_eq, name="SilvanStrategy",
                          line=dict(color="#c0392b", width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=headline_trades["close_date"], y=headline_dd, name="Drawdown",
                          line=dict(color="#c0392b", width=1), fill="tozeroy"), row=2, col=1)
fig.update_layout(template="plotly_dark", height=650, showlegend=False)
fig.show()

In [4]:
pd.DataFrame([compute_metrics(headline_wallet, "Headline (full-period fit)")]).set_index("Label")

,Total %,CAGR %,Sharpe,Sortino,Max DD %,Calmar
Label,,,,,,
Headline (full-period fit),+982.51%,+43.63%,1.48,0.91,-27.77%,28.16


## 2. Parameter fragility — a peak, not a slope

`adx_threshold` and `bb_std` swept on a grid, every other hyperopted parameter held fixed at its optimal value. This is the frozen CSV of that sweep (49 separate `freqtrade backtesting` runs) — see the repo history for the sweep script.

In [5]:
sweep = pd.read_csv(HEATMAP_CSV)
pivot = sweep.pivot(index="adx_threshold", columns="bb_std", values="total_profit_pct")

fig = go.Figure(go.Heatmap(
    z=pivot.values, x=pivot.columns.astype(str), y=pivot.index.astype(str),
    colorscale="RdYlGn", text=pivot.values, texttemplate="%{text:g}",
    colorbar=dict(title="Total profit %"),
))
fig.update_layout(
    title="Total profit % across a local parameter grid (peak at the hyperopted optimum: adx_threshold=27, bb_std=2.225)",
    xaxis_title="bb_std", yaxis_title="adx_threshold",
    template="plotly_dark", height=550,
)
fig.show()

## 3. The walk-forward reveal — Split A

Parameters hyperopted **only** on 2018–2023. Carried forward, completely unmodified, across the full period — the shaded regions mark in-sample (fit here) vs. out-of-sample (never seen by the optimizer).

In [6]:
SPLIT_DATES = {"A": pd.Timestamp("2023-01-01", tz="UTC"), "B": pd.Timestamp("2024-01-01", tz="UTC"), "C": pd.Timestamp("2022-01-01", tz="UTC")}

splitA_data, splitA_wallet, _ = load_backtest(SPLIT_A_ZIP)
splitA_trades = build_trades(splitA_data)
splitA_eq, splitA_dd = equity_from_trades(splitA_trades)

split_date = SPLIT_DATES["A"]
is_mask = splitA_trades["close_date"] < split_date

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3],
                     subplot_titles=["SilvanStrategy — one continuous equity curve, fit only on the in-sample half", "Drawdown %"])
fig.add_vrect(x0=splitA_trades["close_date"].min(), x1=split_date, fillcolor="#2e7d32", opacity=0.12, line_width=0, row=1, col=1)
fig.add_vrect(x0=split_date, x1=splitA_trades["close_date"].max(), fillcolor="#c0392b", opacity=0.12, line_width=0, row=1, col=1)
fig.add_vrect(x0=splitA_trades["close_date"].min(), x1=split_date, fillcolor="#2e7d32", opacity=0.12, line_width=0, row=2, col=1)
fig.add_vrect(x0=split_date, x1=splitA_trades["close_date"].max(), fillcolor="#c0392b", opacity=0.12, line_width=0, row=2, col=1)
fig.add_vline(x=split_date, line_dash="dot", line_color="white", row=1, col=1)
fig.add_vline(x=split_date, line_dash="dot", line_color="white", row=2, col=1)
fig.add_trace(go.Scatter(x=splitA_trades["close_date"], y=splitA_eq, name="Equity",
                          line=dict(color="#e6e6e6", width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=splitA_trades["close_date"], y=splitA_dd, name="Drawdown",
                          line=dict(color="#e6e6e6", width=1), fill="tozeroy"), row=2, col=1)
fig.update_layout(template="plotly_dark", height=650, showlegend=False)
fig.show()

## 4. Three independent splits

A single split invites the objection that the test window just happened to be unlucky. Here are all three, computed from their own frozen continuous run, sliced at each split's own cutoff.

In [7]:
def split_is_oos_metrics(wallet_df, trades, split_date, label):
    w = wallet_df.copy()
    w["date"] = pd.to_datetime(w["date"])

    def metrics_for(mask_wallet, mask_trades, sub_label):
        sub_wallet = w[mask_wallet]
        if len(sub_wallet) < 2:
            return None
        m = compute_metrics(sub_wallet, f"{label} — {sub_label}")
        m["Trades"] = int(mask_trades.sum())
        return m

    is_wallet_mask = w["date"] < split_date
    oos_wallet_mask = w["date"] >= split_date
    is_trades_mask = trades["close_date"] < split_date
    oos_trades_mask = trades["close_date"] >= split_date

    return [
        metrics_for(is_wallet_mask, is_trades_mask, "in-sample"),
        metrics_for(oos_wallet_mask, oos_trades_mask, "out-of-sample"),
    ]


splitB_data, splitB_wallet, _ = load_backtest(SPLIT_B_ZIP)
splitB_trades = build_trades(splitB_data)

splitC_data, splitC_wallet, _ = load_backtest(SPLIT_C_ZIP)
splitC_trades = build_trades(splitC_data)

rows = []
rows += split_is_oos_metrics(splitA_wallet, splitA_trades, SPLIT_DATES["A"], "Split A (train 2018-2023)")
rows += split_is_oos_metrics(splitB_wallet, splitB_trades, SPLIT_DATES["B"], "Split B (train 2018-2024)")
rows += split_is_oos_metrics(splitC_wallet, splitC_trades, SPLIT_DATES["C"], "Split C (train 2018-2022)")

summary = pd.DataFrame([r for r in rows if r is not None]).set_index("Label")
summary

,Total %,CAGR %,Sharpe,Sortino,Max DD %,Calmar,Trades
Label,,,,,,,
Split A (train 2018-2023) — in-sample,+706.46%,+105.62%,2.22,1.57,-19.56%,65.29,901
Split A (train 2018-2023) — out-of-sample,-4.76%,-1.32%,0.09,0.06,-43.42%,-0.16,1075
Split B (train 2018-2024) — in-sample,+225.20%,+35.35%,1.24,0.70,-19.46%,15.55,723
Split B (train 2018-2024) — out-of-sample,-32.78%,-13.76%,-0.43,-0.25,-49.33%,-1.30,576
Split C (train 2018-2022) — in-sample,+296.92%,+106.91%,2.33,1.83,-15.41%,53.20,477
Split C (train 2018-2022) — out-of-sample,+14.68%,+2.97%,0.25,0.17,-54.91%,0.30,1409


In [8]:
labels = ["Split A", "Split B", "Split C"]
is_cagr = [float(summary.loc[f"{l} (train 2018-{y})" + " — in-sample", "CAGR %"].replace("%", "").replace("+", ""))
           for l, y in zip(labels, [2023, 2024, 2022])]
oos_cagr = [float(summary.loc[f"{l} (train 2018-{y})" + " — out-of-sample", "CAGR %"].replace("%", "").replace("+", ""))
            for l, y in zip(labels, [2023, 2024, 2022])]

fig = go.Figure()
fig.add_trace(go.Bar(x=labels, y=is_cagr, name="In-sample CAGR", marker_color="#2e7d32"))
fig.add_trace(go.Bar(x=labels, y=oos_cagr, name="Out-of-sample CAGR", marker_color="#c0392b"))
fig.update_layout(
    title="SilvanStrategy — three independent train/test splits, same collapse every time",
    yaxis_title="CAGR %", barmode="group", template="plotly_dark", height=500,
)
fig.show()

## 5. Takeaway

Same strategy, same code, three different cut points, three different market regimes in the test
window — and the same collapse every time. That's not one unlucky test window, it's what happens
whenever this strategy meets data it wasn't fit on.

Full writeup, the five sins, and the code: [GitHub](https://github.com/arrabyte/freqtrade-strategies) ·
[Backtests, not Signals](https://backtestsnotsignals.substack.com)

*Not financial advice. SilvanStrategy is an educational example and should not be traded.*